# Networking

## Shared Memory
- Multiple processes on the **same machine** access the same region of RAM.
  - Trading engine <-> Risk module
- Communication between processes.

- Faster
  - Threading by default shared memory

## Sockets
- An endpoint for sending or receiving data across a network
- Communication between two machines
  - TCP socket
    - One-to-one
    - Connection-oriented
    - Reliable data transfer
  - UDP socket
    - One-to-many
    - Connectionless (Broadcast protocol)
    - Unreliable data transfer

| Feature                | TCP (Transmission Control Protocol)         | UDP (User Datagram Protocol)             |
|------------------------|----------------------------------------------|-------------------------------------------|
| Connection?            | Yes — requires handshake (connect/accept)   | No — connectionless (sendto/recvfrom)     |
| Reliability            | Reliable (retransmission, ACKs)             | Unreliable (no guarantee)                 |
| Message Order          | Guaranteed ordered delivery                 | No ordering guarantee                     |
| Delivery Guarantee     | Guaranteed delivery                         | May lose packets                          |
| Speed                  | Slower (overhead of reliability)            | Faster (no handshake, no checks)          |
| Message Boundaries     | Stream-based (continuous bytes)             | Message-based (packets preserved)         |
| Overhead               | High                                        | Low                                       |
| Best For               | Downloads, web servers, chat apps           | Streaming, gaming, real-time market data  |
| Python API             | socket.SOCK_STREAM                          | socket.SOCK_DGRAM                         |
| Code Calls             | connect(), accept(), send(), recv()         | sendto(), recvfrom()                      |


| Feature         | Shared Memory                    | Sockets                          |
| --------------- | -------------------------------- | -------------------------------- |
| Speed           | 🚀 Fastest                       | ⚡ Slower (serialization needed)  |
| Scope           | Same machine only                | Local or remote machines         |
| Data            | Raw memory buffers               | Structured messages              |
| Synchronization | Manual                           | Built-in message boundaries      |
| Use Case        | High-frequency trading, robotics | Web servers, distributed systems |


**How does Exchange send orderbook to clients?**

- Multicast UDP Sockets
  - Fast, low latency
  - Scalable to many clients (one-to-many)
  - Market data only, no order submission
    - No guarantee of delivery
    - Message loss acceptable

    -> **Gap detection** and **retransmission requests** (TCP) in client software

### TCP 

**Server**

`bind()`: Bind the socket to an IP address and port number

`listen()`: Listen for incoming connections

`accept()`: Accept a connection from a client

`recv()`: Receive data from the client

In [ ]:
import socket

# 1. Create TCP socket (IPv4 + TCP stream)
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 2. Bind the server to an address/port
server.bind(("localhost", 9000))

# 3. Listen for incoming connections
server.listen()

print("Server waiting for connection...")

# 4. Accept a client connection (blocking)
conn, addr = server.accept()
print("Connected by:", addr)

# 5. Receive data from client
data = conn.recv(1024)
print("Client says:", data.decode())

# 6. Send a reply
conn.send(b"Hello from server")

conn.close()
server.close()

**Client**

`connect()`: Connect to the server

`send()`: Send data to the server

`recv()`: Receive data from the server

In [ ]:
import socket

# 1. Create TCP socket
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 2. Connect to the server
client.connect(("localhost", 9000))

# 3. Send data
client.send(b"Hello from client")

# 4. Receive reply
response = client.recv(1024)
print("Server says:", response.decode())

client.close()

### UDP 

**Server**

`bind()`: Bind the socket to an IP address and port number

`recvfrom()`: Receive data from a client

In [ ]:
import socket

# 1. Create UDP socket
server = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# 2. Bind to address/port
server.bind(("localhost", 9000))

print("UDP server waiting...")

# 3. Receive data (does NOT establish a connection)
data, addr = server.recvfrom(1024)
print("Received:", data.decode())

# 4. Reply using sendto()
server.sendto(b"Hello from server", addr)

**Client**

`sendto()`: Send data to the server

`recvfrom()`: Receive data from the server

In [ ]:
import socket

# 1. Create UDP socket
client = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# 2. Send a message (no connect needed!)
client.sendto(b"Hello from client", ("localhost", 9000))

# 3. Receive response
response, addr = client.recvfrom(1024)
print("Server says:", response.decode())